In [ ]:
import ollama


In [ ]:
from ollama import chat


In [ ]:
import os


In [ ]:
import re


In [ ]:
import numpy as np


In [ ]:
import pandas as pd


In [ ]:
import torch


In [ ]:
from datetime import date


In [ ]:
from typing import List, Literal, Optional


In [ ]:
from pydantic import BaseModel, Field


In [ ]:
from sklearn.metrics import classification_report


In [ ]:
np.random.seed(123)


In [ ]:
set_seed(123)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'set_seed' is not defined



In [ ]:
EventType = Literal[
    "protest",
    "election",
    "policy_change",
    "violence",
    "disaster",
    "other"
]



In [ ]:
import json
import re


In [ ]:
np.random.seed(123)


In [ ]:
set_seed(123)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'set_seed' is not defined



In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed


In [ ]:
set_seed(123)


In [ ]:
GeoPrecision = Literal[
    "country_only",
    "admin1_or_state",
    "city_or_local",
    "unknown"
]



In [ ]:
class EvidenceSpan(BaseModel):
    field: Literal["event_type", "date", "location", "actors", "outcome"]
    quote: str



In [ ]:
class EventExtraction(BaseModel):
    doc_id: str
    event_type: EventType
    event_date_iso: Optional[str] = Field(
        default=None,
        description="ISO date YYYY-MM-DD if available; otherwise null."
    )
    date_is_approximate: bool = Field(
        description="True if the date is estimated/inferred (e.g., 'early April')."
    )



In [ ]:
class EventExtraction(BaseModel):
    doc_id: str
    event_type: EventType
    event_date_iso: Optional[str] = Field(
        default=None,
        description="ISO date YYYY-MM-DD if available; otherwise null."
    )
    date_is_approximate: bool = Field(
        description="True if the date is estimated/inferred (e.g., 'early April')."
    )
    country: Optional[str] = None
    admin1_or_state: Optional[str] = None
    city_or_local: Optional[str] = None
    geo_precision: GeoPrecision
    actors: List[str] = Field(description="Key actors mentioned (individuals, orgs, groups).")
    outcome_summary: Optional[str] = Field(
        default=None,
        description="One-sentence outcome summary (what happened)."
    )
    extraction_confidence: float = Field(
        ge=0.0, le=1.0,
        description="Model self-rated confidence (0 to 1)."
    )
    uncertainty_flags: List[str] = Field(
        description="List of issues that make extraction uncertain (e.g., missing date, vague location)."
    )
    evidence: List[EvidenceSpan] = Field(
        description="Short quotes supporting key fields."
    )



In [ ]:
docs = [
    {"doc_id": "doc_001", "text": "Breaking: Thousands rallied in Santiago on 2026-03-14 demanding pension reform. Police reported minor clashes; 12 were arrested."},
    {"doc_id": "doc_002", "text": "On March 2nd, lawmakers passed the 'Clean Air Act' amendment in the national assembly. Environmental groups praised the vote."},
    {"doc_id": "doc_003", "text": "Election officials said voting will take place next Sunday. Turnout is expected to be high in the capital."},
    {"doc_id": "doc_004", "text": "A 6.2 magnitude earthquake struck near the coastal city overnight, damaging dozens of homes and cutting power to 40,000 residents."},
    {"doc_id": "doc_005", "text": "Witnesses described gunfire outside a nightclub late Friday; at least two people were injured, but details remain unclear."},
    {"doc_id": "doc_006", "text": "The governor announced a new curfew order effective immediately. Critics called it an overreach."},
    {"doc_id": "doc_007", "text": "Early April saw renewed demonstrations in the northern province after fuel prices rose again."},
    {"doc_id": "doc_008", "text": "Floodwaters inundated low-lying neighborhoods; emergency shelters opened at local schools, officials said."},
    {"doc_id": "doc_009", "text": "Opposition leaders met with international observers in Brussels to discuss election monitoring."},
    {"doc_id": "doc_010", "text": "Police said the suspect was arrested after a stabbing in downtown; the mayor urged calm."},
    {"doc_id": "doc_011", "text": "Parliament reversed the prior ban on rideshare apps, citing labor market flexibility."},
    {"doc_id": "doc_012", "text": "A protest was planned for tomorrow, but organizers postponed it due to severe weather warnings."},
    {"doc_id": "doc_013", "text": "Following a landslide, the ministry declared a state of emergency in two districts."},
    {"doc_id": "doc_014", "text": "The court ruling sparked demonstrations across the city center; human rights groups condemned the decision."},
    {"doc_id": "doc_015", "text": "The article mentions reforms and elections in passing but gives no clear time or place."},
]


In [ ]:
docs_df = pd.DataFrame(docs)


In [ ]:
print("\n------------------------------")
print("Input corpus (first 5 docs)")
print("------------------------------")
print(docs_df.head())
print("docs_df shape:", docs_df.shape)



------------------------------
Input corpus (first 5 docs)
------------------------------
    doc_id                                               text
0  doc_001  Breaking: Thousands rallied in Santiago on 202...
1  doc_002  On March 2nd, lawmakers passed the 'Clean Air ...
2  doc_003  Election officials said voting will take place...
3  doc_004  A 6.2 magnitude earthquake struck near the coa...
4  doc_005  Witnesses described gunfire outside a nightclu...
docs_df shape: (15, 2)


In [ ]:
son_template = {
    "doc_id": "doc_XXX",
    "event_type": "other",
    "event_date_iso": None,
    "date_is_approximate": False,
    "country": None,
    "admin1_or_state": None,
    "city_or_local": None,
    "geo_precision": "unknown",
    "actors": [],
    "outcome_summary": None,
    "extraction_confidence": 0.5,
    "uncertainty_flags": [],
    "evidence": [
        {"field": "event_type", "quote": ""},
        {"field": "date", "quote": ""},
        {"field": "location", "quote": ""},
        {"field": "actors", "quote": ""},
        {"field": "outcome", "quote": ""}
    ]
}

model_name = "ollama"
extractions = []


In [ ]:
json_template = {
    "doc_id": "doc_XXX",
    "event_type": "other",
    "event_date_iso": None,
    "date_is_approximate": False,
    "country": None,
    "admin1_or_state": None,
    "city_or_local": None,
    "geo_precision": "unknown",
    "actors": [],
    "outcome_summary": None,
    "extraction_confidence": 0.5,
    "uncertainty_flags": [],
    "evidence": [
        {"field": "event_type", "quote": ""},
        {"field": "date", "quote": ""},
        {"field": "location", "quote": ""},
        {"field": "actors", "quote": ""},
        {"field": "outcome", "quote": ""}
    ]
}

model_name = "ollama"
extractions = []


In [ ]:
model_name = "ollama"


In [ ]:
extractions = []

In [ ]:
prompt = (
    "Task: Extract ONE event record from the text in docs.\n"
    "Output MUST be valid JSON only (no markdown, no extra text).\n"
    "Allowed event_type: protest, election, policy_change, violence, disaster, other.\n"
    "Allowed geo_precision: country_only, admin1_or_state, city_or_local, unknown.\n"
    "If unknown: use null for optional fields, add an uncertainty flag, and lower extraction_confidence.\n"
    "Evidence quotes must be short substrings copied from the text.\n"
)



In [ ]:
result = ollama.generate(model='llama3.1', prompt=prompt)
print("Summary:", result['response'])

Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\ollama\_client.py", line 262, in generate
    return self._request(
           ^^^^^^^^^^^^^^
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\ollama\_client.py", line 189, in _request
    return cls(**self._request_raw(*args, **kwargs).json())
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\

In [ ]:
eval_df = gold.merge(extractions_df[["doc_id", "event_type"]], on="doc_id", how="left")
eval_df = eval_df.rename(columns={"event_type": "event_type_pred"})
eval_df["event_type_pred"] = eval_df["event_type_pred"].fillna("MISSING")


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'gold' is not defined



In [ ]:
gold = pd.DataFrame([
    {"doc_id": "doc_001", "event_type_gold": "protest"},
    {"doc_id": "doc_002", "event_type_gold": "policy_change"},
    {"doc_id": "doc_003", "event_type_gold": "election"},
    {"doc_id": "doc_004", "event_type_gold": "disaster"},
    {"doc_id": "doc_005", "event_type_gold": "violence"},
    {"doc_id": "doc_006", "event_type_gold": "policy_change"},
    {"doc_id": "doc_007", "event_type_gold": "protest"},
    {"doc_id": "doc_008", "event_type_gold": "disaster"},
])


In [ ]:
eval_df = gold.merge(extractions_df[["doc_id", "event_type"]], on="doc_id", how="left")


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'extractions_df' is not defined. Did you mean: 'extractions'?



In [ ]:
extractions_df = pd.DataFrame(extractions)


In [ ]:
eval_df = gold.merge(extractions_df[["doc_id", "event_type"]], on="doc_id", how="left")


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\frame.py", line 4384, in __getitem__
    indexer = self.columns._get_indexer_strict(key, "columns")[1]
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\indexes\base.py", line 6302, in _get_indexer_strict
    self._raise_if_missing(keyarr, indexer, axis_name)
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\L

In [ ]:
EventType = Literal[




Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1
    EventType = Literal[
                       ^
SyntaxError: '[' was never closed



In [ ]:
EventType = Literal[
    "protest",
    "election",
    "policy_change",
    "violence",
    "disaster",
    "other"
]



In [ ]:
GeoPrecision = Literal[
    "country_only",
    "admin1_or_state",
    "city_or_local",
    "unknown"
]



In [ ]:
class EvidenceSpan(BaseModel):
    field: Literal["event_type", "date", "location", "actors", "outcome"]
    quote: str



In [ ]:
quote: str


In [ ]:
class EventExtraction(BaseModel):
    doc_id: str
    event_type: EventType
    event_date_iso: Optional[str] = Field(
        default=None,
        description="ISO date YYYY-MM-DD if available; otherwise null."
    )
    date_is_approximate: bool = Field(
        description="True if the date is estimated/inferred (e.g., 'early April')."
    )
    country: Optional[str] = None
    admin1_or_state: Optional[str] = None
    city_or_local: Optional[str] = None
    geo_precision: GeoPrecision
    actors: List[str] = Field(description="Key actors mentioned (individuals, orgs, groups).")
    outcome_summary: Optional[str] = Field(
        default=None,
        description="One-sentence outcome summary (what happened)."
    )
    extraction_confidence: float = Field(
        ge=0.0, le=1.0,
        description="Model self-rated confidence (0 to 1)."
    )
    uncertainty_flags: List[str] = Field(
        description="List of issues that make extraction uncertain (e.g., missing date, vague location)."
    )
    evidence: List[EvidenceSpan] = Field(
        description="Short quotes supporting key fields."
    )



In [ ]:
docs = [
    {"doc_id": "doc_001", "text": "Breaking: Thousands rallied in Santiago on 2026-03-14 demanding pension reform. Police reported minor clashes; 12 were arrested."},
    {"doc_id": "doc_002", "text": "On March 2nd, lawmakers passed the 'Clean Air Act' amendment in the national assembly. Environmental groups praised the vote."},
    {"doc_id": "doc_003", "text": "Election officials said voting will take place next Sunday. Turnout is expected to be high in the capital."},
    {"doc_id": "doc_004", "text": "A 6.2 magnitude earthquake struck near the coastal city overnight, damaging dozens of homes and cutting power to 40,000 residents."},
    {"doc_id": "doc_005", "text": "Witnesses described gunfire outside a nightclub late Friday; at least two people were injured, but details remain unclear."},
    {"doc_id": "doc_006", "text": "The governor announced a new curfew order effective immediately. Critics called it an overreach."},
    {"doc_id": "doc_007", "text": "Early April saw renewed demonstrations in the northern province after fuel prices rose again."},
    {"doc_id": "doc_008", "text": "Floodwaters inundated low-lying neighborhoods; emergency shelters opened at local schools, officials said."},
    {"doc_id": "doc_009", "text": "Opposition leaders met with international observers in Brussels to discuss election monitoring."},
    {"doc_id": "doc_010", "text": "Police said the suspect was arrested after a stabbing in downtown; the mayor urged calm."},
    {"doc_id": "doc_011", "text": "Parliament reversed the prior ban on rideshare apps, citing labor market flexibility."},
    {"doc_id": "doc_012", "text": "A protest was planned for tomorrow, but organizers postponed it due to severe weather warnings."},
    {"doc_id": "doc_013", "text": "Following a landslide, the ministry declared a state of emergency in two districts."},
    {"doc_id": "doc_014", "text": "The court ruling sparked demonstrations across the city center; human rights groups condemned the decision."},
    {"doc_id": "doc_015", "text": "The article mentions reforms and elections in passing but gives no clear time or place."},
]


In [ ]:
docs_df = pd.DataFrame(docs)


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Input corpus (first 5 docs)")


Input corpus (first 5 docs)


In [ ]:
print("------------------------------")


------------------------------


In [ ]:
print(docs_df.head())


    doc_id                                               text
0  doc_001  Breaking: Thousands rallied in Santiago on 202...
1  doc_002  On March 2nd, lawmakers passed the 'Clean Air ...
2  doc_003  Election officials said voting will take place...
3  doc_004  A 6.2 magnitude earthquake struck near the coa...
4  doc_005  Witnesses described gunfire outside a nightclu...


In [ ]:
print("docs_df shape:", docs_df.shape)


docs_df shape: (15, 2)


In [ ]:
json_template = {
    "doc_id": "doc_XXX",
    "event_type": "other",
    "event_date_iso": None,
    "date_is_approximate": False,
    "country": None,
    "admin1_or_state": None,
    "city_or_local": None,
    "geo_precision": "unknown",
    "actors": [],
    "outcome_summary": None,
    "extraction_confidence": 0.5,
    "uncertainty_flags": [],
    "evidence": [
        {"field": "event_type", "quote": ""},
        {"field": "date", "quote": ""},
        {"field": "location", "quote": ""},
        {"field": "actors", "quote": ""},
        {"field": "outcome", "quote": ""}
    ]
}


In [ ]:
system_instructions = (
    "Task: Extract ONE event record from the text in docs_df.\n"
    "Output MUST be valid JSON only (no markdown, no extra text).\n"
    "Allowed event_type: protest, election, policy_change, violence, disaster, other.\n"
    "Allowed geo_precision: country_only, admin1_or_state, city_or_local, unknown.\n"
    "If unknown: use null for optional fields, add an uncertainty flag, and lower extraction_confidence.\n"
    "Evidence quotes must be short substrings copied from the text.\n"
)



In [ ]:
model_name = "google/flan-t5-small"


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Loading tokenizer + model")


Loading tokenizer + model


In [ ]:
print("------------------------------")


------------------------------


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:
use_gpu = torch.cuda.is_available()


In [ ]:
device = torch.device("cuda") if use_gpu else torch.device("cpu")


In [ ]:
model = model.to(device)


In [ ]:
model = model.to(device)


In [ ]:
print("Model:", model_name)


Model: google/flan-t5-small


In [ ]:
print("Device:", device)


Device: cpu


In [ ]:
extractions = []


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Running LOCAL LLM extraction (one doc at a time)")


Running LOCAL LLM extraction (one doc at a time)


In [ ]:
print("Running LOCAL LLM extraction (one doc at a time)")


Running LOCAL LLM extraction (one doc at a time)


In [ ]:
for i in range(len(docs_df)):
    doc_id = docs_df.loc[i, "doc_id"]
    text = docs_df.loc[i, "text"]
    prompt = (
        f"{system_instructions}\n"
        f"JSON template:\n{json.dumps(json_template, ensure_ascii=False)}\n\n"
        f"Document ID: {doc_id}\n"
        f"Text: {text}\n\n"
        "Return JSON only."
    )
    # 1) Tokenize (explicit)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    # 2) Generate (explicit)
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False
        )
    # 3) Decode (explicit)
    out_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    # 4) Recover JSON substring (explicit)
    match = re.search(r"\{.*\}", out_text, flags=re.DOTALL)
    json_str = match.group(0) if match else out_text
    # 5) Validate with Pydantic (explicit)
    # Pydantic v2:
    parse_ok = True
    parse_error = ""
    try:
        extracted_obj = EventExtraction.model_validate_json(json_str)
        extra_dict = extracted_obj.model_dump()
    except Exception as e:
        parse_ok = False
        parse_error = str(e)
        # Fallback record (keeps pipeline running)
        extra_dict = {
            "doc_id": doc_id,
            "event_type": "other",
            "event_date_iso": None,
            "date_is_approximate": False,
            "country": None,
            "admin1_or_state": None,
            "city_or_local": None,
            "geo_precision": "unknown",
            "actors": [],
            "outcome_summary": None,
            "extraction_confidence": 0.0,
            "uncertainty_flags": ["parse_failed_local_model_output"],
            "evidence": [
                {"field": "event_type", "quote": ""},
                {"field": "date", "quote": ""},
                {"field": "location", "quote": ""},
                {"field": "actors", "quote": ""},
                {"field": "outcome", "quote": ""}
            ]
        }



In [ ]:
extra_dict["raw_text"] = text
extra_dict["local_model_raw_output"] = out_text
extra_dict["parse_ok"] = parse_ok
extra_dict["parse_error"] = parse_error
# 7) Flatten list fields for CSV (explicit)

extra_dict["evidence_json"] = json.dumps(extra_dict["evidence"], ensure_ascii=False)
extra_dict["uncertainty_flags_json"] = json.dumps(extra_dict["uncertainty_flags"], ensure_ascii=False)
extra_dict.pop("evidence")
extra_dict.pop("uncertainty_flags")
extractions.append(extra_dict)


In [ ]:
extra_dict["local_model_raw_output"] = out_text


In [ ]:
extra_dict["parse_ok"] = parse_ok


In [ ]:
extra_dict["parse_error"] = parse_error


In [ ]:
extra_dict["evidence_json"] = json.dumps(extra_dict["evidence"], ensure_ascii=False)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
KeyError: 'evidence'



In [ ]:
extra_dict["uncertainty_flags_json"] = json.dumps(extra_dict["uncertainty_flags"], ensure_ascii=False)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
KeyError: 'uncertainty_flags'



In [ ]:
extra_dict.pop("evidence")


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
KeyError: 'evidence'



In [ ]:
extra_dict.pop("uncertainty_flags")


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
KeyError: 'uncertainty_flags'



In [ ]:
extractions.append(extra_dict)


In [ ]:
extractions_df = pd.DataFrame(extractions)


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Extracted records (first 5 rows)")


Extracted records (first 5 rows)


In [ ]:
print("------------------------------")


------------------------------


In [ ]:
print(extractions_df.head())


    doc_id  ...               uncertainty_flags_json
0  doc_015  ...  ["parse_failed_local_model_output"]
1  doc_015  ...  ["parse_failed_local_model_output"]

[2 rows x 17 columns]


In [ ]:
print("extractions_df shape:", extractions_df.shape)


extractions_df shape: (2, 17)


In [ ]:
os.makedirs("outputs", exist_ok=True)


In [ ]:
extractions_df.to_csv("outputs/extractions_raw.csv", index=False)


In [ ]:
extractions_df["extraction_confidence"] = pd.to_numeric(extractions_df["extraction_confidence"], errors="coerce")


In [ ]:
extractions_df["flag_parse_failed"] = ~extractions_df["parse_ok"]


In [ ]:
extractions_df["flag_low_confidence"] = extractions_df["extraction_confidence"] < 0.70


In [ ]:
extractions_df["flag_missing_date"] = extractions_df["event_date_iso"].isna()


In [ ]:
extractions_df["flag_missing_country"] = extractions_df["country"].isna()


In [ ]:
extractions_df["flag_geo_unknown"] = extractions_df["geo_precision"].isin(["unknown", "country_only"])


In [ ]:
flag_cols = [
    "flag_parse_failed",
    "flag_low_confidence",
    "flag_missing_date",
    "flag_missing_country",
    "flag_geo_unknown"
]


In [ ]:
extractions_df["needs_human_review"] = extractions_df[flag_cols].any(axis=1)


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Review flag counts")


Review flag counts


In [ ]:
print("------------------------------")


------------------------------


In [ ]:
print(extractions_df[flag_cols + ["needs_human_review"]].sum(numeric_only=True))


flag_parse_failed       2
flag_low_confidence     2
flag_missing_date       2
flag_missing_country    2
flag_geo_unknown        2
needs_human_review      2
dtype: int64


In [ ]:
extractions_df.to_csv("outputs/extractions_with_flags.csv", index=False)


In [ ]:
audit_random_n = 5


In [ ]:
audit_random = extractions_df.sample(n=audit_random_n, random_state=123)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\generic.py", line 6000, in sample
    sampled_indices = sample.sample(obj_len, size, replace, weights, rs)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\sample.py", line 161, in sample
    return random_state.choice(obj_len, size=size, replace=replace, p=weights).astype(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
audit_flagged = extractions_df[extractions_df["needs_human_review"]].copy()


In [ ]:
audit_flagged = extractions_df[extractions_df["needs_human_review"]].copy()


In [ ]:
audit_sheet = pd.concat([audit_random, audit_flagged], ignore_index=True).drop_duplicates(subset=["doc_id"])


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'audit_random' is not defined. Did you mean: 'audit_random_n'?



In [ ]:
audit_sheet = audit_sheet.sort_values("doc_id").reset_index(drop=True)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'audit_sheet' is not defined



In [ ]:
audit_sheet["human_is_correct"] = ""


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'audit_sheet' is not defined



In [ ]:
audit_sheet["human_correct_event_type"] = ""


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'audit_sheet' is not defined



In [ ]:
audit_sheet["human_correct_event_type"] = ""


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'audit_sheet' is not defined



In [ ]:
audit_sheet["human_correct_date_iso"] = ""


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'audit_sheet' is not defined



In [ ]:
audit_sheet["failure_mode"] = ""


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'audit_sheet' is not defined



In [ ]:
gold = pd.DataFrame([
    {"doc_id": "doc_001", "event_type_gold": "protest"},
    {"doc_id": "doc_002", "event_type_gold": "policy_change"},
    {"doc_id": "doc_003", "event_type_gold": "election"},
    {"doc_id": "doc_004", "event_type_gold": "disaster"},
    {"doc_id": "doc_005", "event_type_gold": "violence"},
    {"doc_id": "doc_006", "event_type_gold": "policy_change"},
    {"doc_id": "doc_007", "event_type_gold": "protest"},
    {"doc_id": "doc_008", "event_type_gold": "disaster"},
])


In [ ]:
eval_df = gold.merge(extractions_df[["doc_id", "event_type"]], on="doc_id", how="left")


In [ ]:
eval_df = eval_df.rename(columns={"event_type": "event_type_pred"})

In [ ]:
eval_df["event_type_pred"] = eval_df["event_type_pred"].fillna("MISSING")


In [ ]:
print("\n------------------------------")
print("Evaluation table (gold vs predicted)")
print("------------------------------")
print(eval_df)



------------------------------
Evaluation table (gold vs predicted)
------------------------------
    doc_id event_type_gold event_type_pred
0  doc_001         protest         MISSING
1  doc_002   policy_change         MISSING
2  doc_003        election         MISSING
3  doc_004        disaster         MISSING
4  doc_005        violence         MISSING
5  doc_006   policy_change         MISSING
6  doc_007         protest         MISSING
7  doc_008        disaster         MISSING


In [ ]:
extractions_df.to_csv("C:Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/demo/outputs/extractions_raw.csv", index=False)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'extractions_df' is not defined



In [ ]:
import os


In [ ]:
import json


In [ ]:
import re


In [ ]:
import numpy as np


In [ ]:
import pandas as pd


In [ ]:
import torch


In [ ]:
from datetime import date


In [ ]:
from typing import List, Literal, Optional


In [ ]:
from pydantic import BaseModel, Field


In [ ]:
from sklearn.metrics import classification_report


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed


In [ ]:
np.random.seed(123)


In [ ]:
set_seed(123)


In [ ]:
EventType = Literal[
    "protest",
    "election",
    "policy_change",
    "violence",
    "disaster",
    "other"
]



In [ ]:
GeoPrecision = Literal[
    "country_only",
    "admin1_or_state",
    "city_or_local",
    "unknown"
]


In [ ]:
class EvidenceSpan(BaseModel):
    field: Literal["event_type", "date", "location", "actors", "outcome"]
    quote: str



In [ ]:
class EventExtraction(BaseModel):
    doc_id: str
    # NOTE (teaching/demo setting):
    # For local small models, strict JSON/schema enforcement often yields empty outputs.
    # We therefore provide safe defaults so the pipeline can produce "messy but usable"
    # records while still flagging uncertainty. In a production setting, tighten these
    # requirements and fail fast.
    event_type: EventType = "other"
    event_date_iso: Optional[str] = Field(
        default=None,
        description="ISO date YYYY-MM-DD if available; otherwise null."
    )
    date_is_approximate: bool = Field(
        default=True,
        description="True if the date is estimated/inferred (e.g., 'early April')."
    )
    country: Optional[str] = None
    admin1_or_state: Optional[str] = None
    city_or_local: Optional[str] = None
    geo_precision: GeoPrecision = "unknown"
    actors: List[str] = Field(default_factory=list, description="Key actors mentioned (individuals, orgs, groups).")
    outcome_summary: Optional[str] = Field(
        default=None,
        description="One-sentence outcome summary (what happened)."
    )
    extraction_confidence: float = Field(
        default=0.2, ge=0.0, le=1.0,
        description="Model self-rated confidence (0 to 1)."
    )
    uncertainty_flags: List[str] = Field(
        default_factory=list,
        description="List of issues that make extraction uncertain (e.g., missing date, vague location)."
    )
    evidence: List[EvidenceSpan] = Field(
        default_factory=list,
        description="Short quotes supporting each extracted field (if available)."
    )



In [ ]:
docs = [
    {"doc_id": "doc_001", "text": "Breaking: Thousands rallied in Santiago on 2026-03-14 demanding pension reform. Police reported minor clashes; 12 were arrested."},
    {"doc_id": "doc_002", "text": "On March 2nd, lawmakers passed the 'Clean Air Act' amendment in the national assembly. Environmental groups praised the vote."},
    {"doc_id": "doc_003", "text": "Election officials said voting will take place next Sunday. Turnout is expected to be high in the capital."},
    {"doc_id": "doc_004", "text": "A 6.2 magnitude earthquake struck near the coastal city overnight, damaging dozens of homes and cutting power to 40,000 residents."},
    {"doc_id": "doc_005", "text": "Witnesses described gunfire outside a nightclub late Friday; at least two people were injured, but details remain unclear."},
    {"doc_id": "doc_006", "text": "The governor announced a new curfew order effective immediately. Critics called it an overreach."},
    {"doc_id": "doc_007", "text": "Early April saw renewed demonstrations in the northern province after fuel prices rose again."},
    {"doc_id": "doc_008", "text": "Floodwaters inundated low-lying neighborhoods; emergency shelters opened at local schools, officials said."},
    {"doc_id": "doc_009", "text": "Opposition leaders met with international observers in Brussels to discuss election monitoring."},
    {"doc_id": "doc_010", "text": "Police said the suspect was arrested after a stabbing in downtown; the mayor urged calm."},
    {"doc_id": "doc_011", "text": "Parliament reversed the prior ban on rideshare apps, citing labor market flexibility."},
    {"doc_id": "doc_012", "text": "A protest was planned for tomorrow, but organizers postponed it due to severe weather warnings."},
    {"doc_id": "doc_013", "text": "Following a landslide, the ministry declared a state of emergency in two districts."},
    {"doc_id": "doc_014", "text": "The court ruling sparked demonstrations across the city center; human rights groups condemned the decision."},
    {"doc_id": "doc_015", "text": "The article mentions reforms and elections in passing but gives no clear time or place."},
]


In [ ]:
docs_df = pd.DataFrame(docs)


In [ ]:
print("\n------------------------------")
print("Input corpus (first 5 docs)")
print("------------------------------")
print(docs_df.head())
print("docs_df shape:", docs_df.shape)



------------------------------
Input corpus (first 5 docs)
------------------------------
    doc_id                                               text
0  doc_001  Breaking: Thousands rallied in Santiago on 202...
1  doc_002  On March 2nd, lawmakers passed the 'Clean Air ...
2  doc_003  Election officials said voting will take place...
3  doc_004  A 6.2 magnitude earthquake struck near the coa...
4  doc_005  Witnesses described gunfire outside a nightclu...
docs_df shape: (15, 2)


In [ ]:
json_template = {
    "doc_id": "doc_XXX",
    "event_type": "other",
    "event_date_iso": None,
    "date_is_approximate": False,
    "country": None,
    "admin1_or_state": None,
    "city_or_local": None,
    "geo_precision": "unknown",
    "actors": [],
    "outcome_summary": None,
    "extraction_confidence": 0.5,
    "uncertainty_flags": [],
    "evidence": [
        {"field": "event_type", "quote": ""},
        {"field": "date", "quote": ""},
        {"field": "location", "quote": ""},
        {"field": "actors", "quote": ""},
        {"field": "outcome", "quote": ""}
    ]
}



In [ ]:
system_instructions = (
    "Task: Extract ONE event record from the text.\n"
    "Return EXACTLY the following 9 lines, one per line, in the format key: value\n"
    "Use empty value if unknown.\n"
    "\n"
    "event_type: protest|election|policy_change|violence|disaster|other\n"
    "event_date_iso: YYYY-MM-DD\n"
    "date_is_approximate: true|false\n"
    "country:\n"
    "admin1_or_state:\n"
    "city_or_local:\n"
    "geo_precision: country_only|admin1_or_state|city_or_local|unknown\n"
    "actors: comma-separated list\n"
    "outcome_summary: one sentence\n"
    "\n"
    "Do not output anything else.\n"
)



In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"


In [ ]:
print("\n------------------------------")
print("Loading tokenizer + model")
print("------------------------------")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token




------------------------------
Loading tokenizer + model
------------------------------


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)



In [ ]:
use_gpu = torch.cuda.is_available()


In [ ]:
device = torch.device("cuda") if use_gpu else torch.device("cpu")


In [ ]:
model = model.to(device)


In [ ]:
print("Model:", model_name)


Model: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
print("Device:", device)


Device: cpu


In [ ]:
extractions = []


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Running LOCAL LLM extraction (one doc at a time)")


Running LOCAL LLM extraction (one doc at a time)


In [ ]:
print("------------------------------")


------------------------------


In [ ]:
for i in range(len(docs_df)):
    doc_id = docs_df.loc[i, "doc_id"]
    text = docs_df.loc[i, "text"]
    prompt = (
        f"{system_instructions}\n"
        f"Document ID: {doc_id}\n"
        f"Text: {text}\n"
    )
    # 1) Tokenize (explicit)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    # 2) Generate (explicit)
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )
    # 3) Decode (explicit)
    out_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    # 4) Parse labeled key:value lines (best-effort)
    parse_ok = True
    parse_error = ""
    parse_flags: List[str] = []
    allowed_keys = {
        "event_type",
        "event_date_iso",
        "date_is_approximate",
        "country",
        "admin1_or_state",
        "city_or_local",
        "geo_precision",
        "actors",
        "outcome_summary"
    }
    lines = [ln.strip() for ln in out_text.splitlines() if ":" in ln]
    kv = {}
    for ln in lines:
        k, v = ln.split(":", 1)
        k_norm = k.strip().lower()
        if k_norm in allowed_keys:
            kv[k_norm] = v.strip()
    if len(kv) == 0:
        parse_ok = False
        parse_error = "no_key_value_lines_found"
        parse_flags.append("parse_failed_local_model_output")
    event_type = (kv.get("event_type", "other") or "other").strip().lower()
    if event_type not in {"protest", "election", "policy_change", "violence", "disaster", "other"}:
        parse_flags.append("invalid_event_type_from_model")
        event_type = "other"
    event_date_iso = (kv.get("event_date_iso", "") or "").strip() or None
    date_is_approx_raw = (kv.get("date_is_approximate", "") or "").strip().lower()
    if date_is_approx_raw in {"true", "false"}:
        date_is_approximate = (date_is_approx_raw == "true")
    else:
        parse_flags.append("date_is_approximate_missing_or_invalid")
        date_is_approximate = True
    country = (kv.get("country", "") or "").strip() or None
    admin1_or_state = (kv.get("admin1_or_state", "") or "").strip() or None
    city_or_local = (kv.get("city_or_local", "") or "").strip() or None
    geo_precision = (kv.get("geo_precision", "unknown") or "unknown").strip().lower()
    if geo_precision not in {"country_only", "admin1_or_state", "city_or_local", "unknown"}:
        parse_flags.append("invalid_geo_precision_from_model")
        geo_precision = "unknown"
    actors_raw = (kv.get("actors", "") or "").strip()
    actors = [a.strip() for a in actors_raw.split(",") if a.strip()] if actors_raw else []
    outcome_summary = (kv.get("outcome_summary", "") or "").strip() or None
    extracted_obj = EventExtraction(
        doc_id=doc_id,
        event_type=event_type,
        event_date_iso=event_date_iso,
        date_is_approximate=date_is_approximate,
        country=country,
        admin1_or_state=admin1_or_state,
        city_or_local=city_or_local,
        geo_precision=geo_precision,
        actors=actors,
        outcome_summary=outcome_summary,
        extraction_confidence=0.35 if parse_ok else 0.2,
        uncertainty_flags=parse_flags,
        evidence=[]
    )
    extra_dict = extracted_obj.model_dump()
    # 5) Attach trace fields (explicit)
    extra_dict["raw_text"] = text
    extra_dict["local_model_raw_output"] = out_text
    extra_dict["parse_ok"] = parse_ok
    extra_dict["parse_error"] = parse_error
    # 6) Flatten list fields for CSV (explicit)
    extra_dict["evidence_json"] = json.dumps(extra_dict["evidence"], ensure_ascii=False)
    extra_dict["uncertainty_flags_json"] = json.dumps(extra_dict["uncertainty_flags"], ensure_ascii=False)
    extra_dict.pop("evidence")
    extra_dict.pop("uncertainty_flags")
    extractions.append(extra_dict)



In [ ]:
extractions_df = pd.DataFrame(extractions)


In [ ]:
print("\n------------------------------")
print("Extracted records (first 5 rows)")
print("------------------------------")
print(extractions_df.head())
print("extractions_df shape:", extractions_df.shape)



------------------------------
Extracted records (first 5 rows)
------------------------------
    doc_id  ...                             uncertainty_flags_json
0  doc_001  ...                  ["invalid_event_type_from_model"]
1  doc_002  ...                                                 []
2  doc_003  ...  ["invalid_event_type_from_model", "date_is_app...
3  doc_004  ...                                                 []
4  doc_005  ...  ["invalid_event_type_from_model", "date_is_app...

[5 rows x 17 columns]
extractions_df shape: (15, 17)


In [ ]:
extractions_df.to_csv("C:Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/extractions_raw.csv", index=False)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\generic.py", line 3988, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\io\formats\format.py", line 1025, in to_csv
    csv_formatter.save()
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\io\formats\csvs.py", li

In [ ]:
extractions_df.to_csv("C:/Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/extractions_raw.csv", index=False)


In [ ]:
extractions_df["flag_parse_failed"] = ~extractions_df["parse_ok"]
extractions_df["flag_low_confidence"] = extractions_df["extraction_confidence"] < 0.70
extractions_df["flag_missing_date"] = extractions_df["event_date_iso"].isna()
extractions_df["flag_missing_country"] = extractions_df["country"].isna()
extractions_df["flag_geo_unknown"] = extractions_df["geo_precision"].isin(["unknown", "country_only"])


In [ ]:
extractions_df["flag_low_confidence"] = extractions_df["extraction_confidence"] < 0.70


In [ ]:
extractions_df["flag_missing_date"] = extractions_df["event_date_iso"].isna()


In [ ]:
extractions_df["flag_missing_country"] = extractions_df["country"].isna()


In [ ]:
extractions_df["flag_geo_unknown"] = extractions_df["geo_precision"].isin(["unknown", "country_only"])


In [ ]:
flag_cols = [
    "flag_parse_failed",
    "flag_low_confidence",
    "flag_missing_date",
    "flag_missing_country",
    "flag_geo_unknown"
]


In [ ]:
extractions_df["needs_human_review"] = extractions_df[flag_cols].any(axis=1)


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Review flag counts")


Review flag counts


In [ ]:
print("------------------------------")


------------------------------


In [ ]:
print(extractions_df[flag_cols + ["needs_human_review"]].sum(numeric_only=True))


flag_parse_failed        0
flag_low_confidence     15
flag_missing_date        0
flag_missing_country     0
flag_geo_unknown         6
needs_human_review      15
dtype: int64


In [ ]:
extractions_df.to_csv("C:/Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/extractions_with_flags.csv", index=False)


In [ ]:
audit_random_n = 5


In [ ]:
audit_random = extractions_df.sample(n=audit_random_n, random_state=123)


In [ ]:
audit_flagged = extractions_df[extractions_df["needs_human_review"]].copy()


In [ ]:
audit_sheet = pd.concat([audit_random, audit_flagged], ignore_index=True).drop_duplicates(subset=["doc_id"])


In [ ]:
audit_sheet = audit_sheet.sort_values("doc_id").reset_index(drop=True)


In [ ]:
audit_sheet["human_is_correct"] = ""


In [ ]:
audit_sheet["human_correct_event_type"] = ""


In [ ]:
audit_sheet["human_correct_date_iso"] = ""


In [ ]:
audit_sheet["human_correct_location"] = ""


In [ ]:
audit_sheet["failure_mode"] = ""


In [ ]:
audit_sheet["reviewer_notes"] = ""


In [ ]:
audit_sheet.to_csv("outputs/human_audit_sheet.csv", index=False)


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Wrote outputs/human_audit_sheet.csv")


Wrote outputs/human_audit_sheet.csv


In [ ]:
print("------------------------------")


------------------------------


In [ ]:
gold = pd.DataFrame([
    {"doc_id": "doc_001", "event_type_gold": "protest"},
    {"doc_id": "doc_002", "event_type_gold": "policy_change"},
    {"doc_id": "doc_003", "event_type_gold": "election"},
    {"doc_id": "doc_004", "event_type_gold": "disaster"},
    {"doc_id": "doc_005", "event_type_gold": "violence"},
    {"doc_id": "doc_006", "event_type_gold": "policy_change"},
    {"doc_id": "doc_007", "event_type_gold": "protest"},
    {"doc_id": "doc_008", "event_type_gold": "disaster"},
])


In [ ]:
eval_df = gold.merge(extractions_df[["doc_id", "event_type"]], on="doc_id", how="left")


In [ ]:
eval_df = eval_df.rename(columns={"event_type": "event_type_pred"})


In [ ]:
eval_df["event_type_pred"] = eval_df["event_type_pred"].fillna("MISSING")


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Evaluation table (gold vs predicted)")


Evaluation table (gold vs predicted)


In [ ]:
print("------------------------------")


------------------------------


In [ ]:
print(eval_df)


    doc_id event_type_gold event_type_pred
0  doc_001         protest           other
1  doc_002   policy_change   policy_change
2  doc_003        election           other
3  doc_004        disaster        disaster
4  doc_005        violence           other
5  doc_006   policy_change           other
6  doc_007         protest         protest
7  doc_008        disaster        disaster


In [ ]:
audit_sheet.to_csv("C:/Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/human_audit_sheet.csv", index=False)


In [ ]:
eval_df.to_csv("C:/Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/eval_table.csv", index=False)


In [ ]:
print("\n------------------------------")



------------------------------


In [ ]:
print("Classification report (event_type)")


Classification report (event_type)


In [ ]:
print("------------------------------")


------------------------------


In [ ]:
print(classification_report(eval_df["event_type_gold"], eval_df["event_type_pred"], zero_division=0))


               precision    recall  f1-score   support

     disaster       1.00      1.00      1.00         2
     election       0.00      0.00      0.00         1
        other       0.00      0.00      0.00         0
policy_change       1.00      0.50      0.67         2
      protest       1.00      0.50      0.67         2
     violence       0.00      0.00      0.00         1

     accuracy                           0.50         8
    macro avg       0.50      0.33      0.39         8
 weighted avg       0.75      0.50      0.58         8



In [ ]:
report= classification_report(eval_df["event_type_gold"], eval_df["event_type_pred"], zero_division=0)


In [ ]:
report.to_csv("C:/Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/event_type_class_rpt.csv", index=False)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
AttributeError: 'str' object has no attribute 'to_csv'



In [ ]:
report_df= classification_report(eval_df["event_type_gold"], eval_df["event_type_pred"], zero_division=0)


In [ ]:
report_df.to_csv("C:/Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/event_type_class_rpt.csv", index=False)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
AttributeError: 'str' object has no attribute 'to_csv'



In [ ]:
from io import StringIO


In [ ]:
report_df = """
              precision    recall  f1-score   support
     disaster       1.00      1.00      1.00         2
     election       0.00      0.00      0.00         1
        other       0.00      0.00      0.00         0
policy_change       1.00      0.50      0.67         2
      protest       1.00      0.50      0.67         2
     violence       0.00      0.00      0.00         1
     accuracy                           0.50         8
    macro avg       0.50      0.33      0.39         8
 weighted avg       0.75      0.50      0.58         8
"""

df = pd.read_fwf(StringIO(report_df))
report_df.to_csv("C:/Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/event_type_class_rpt.csv", index=False)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 15, in <module>
AttributeError: 'str' object has no attribute 'to_csv'



In [ ]:
report_df = """
              precision    recall  f1-score   support
     disaster       1.00      1.00      1.00         2
     election       0.00      0.00      0.00         1
        other       0.00      0.00      0.00         0
policy_change       1.00      0.50      0.67         2
      protest       1.00      0.50      0.67         2
     violence       0.00      0.00      0.00         1
     accuracy                           0.50         8
    macro avg       0.50      0.33      0.39         8
 weighted avg       0.75      0.50      0.58         8
"""



In [ ]:
df = pd.read_fwf(StringIO(report_df))


In [ ]:
report_df.to_csv("C:/Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/event_type_class_rpt.csv", index=False)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.2.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
AttributeError: 'str' object has no attribute 'to_csv'



In [ ]:
report = """
              precision    recall  f1-score   support
     disaster       1.00      1.00      1.00         2
     election       0.00      0.00      0.00         1
        other       0.00      0.00      0.00         0
policy_change       1.00      0.50      0.67         2
      protest       1.00      0.50      0.67         2
     violence       0.00      0.00      0.00         1
     accuracy                           0.50         8
    macro avg       0.50      0.33      0.39         8
 weighted avg       0.75      0.50      0.58         8
"""



In [ ]:
report_df = pd.read_fwf(StringIO(report))


In [ ]:
report_df.to_csv("C:/Users/karra/Desktop/Coding_work/soda_501/07_llm_human_interface/outputs/event_type_class_rpt.csv", index=False)
